In [ ]:
from pathlib import Path
from pprint import pprint

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
PROJECT_ROOT = Path().resolve().parent
REPORT_PATH = PROJECT_ROOT / "report-2026-05-14.joblib"

In [ ]:
data = joblib.load(REPORT_PATH)
len(data)

1044797

In [ ]:
pprint(data[0])

{'cct_true': 0.55,
 'cct_weighted_global': 0.6463037968913501,
 'cct_weighted_per_location': 0.585490986265348,
 'crit_gen_true': 'g 03',
 'has_crit_gen_prediction': True,
 'has_location_prediction': True,
 'location_neighbor_count': 100,
 'location_true': 'bus 10',
 'location_weight_mass': 0.17124912541676224,
 'n_eff': 580.1423243742473,
 'n_neighbors': 584,
 'prediction_summary': ReportSummary(cct_weighted=0.6463037968913501, cct_weighted_per_location={'Bus 10': 0.585490986265348, 'Bus 11': 0.683040977495455, 'Bus 13': 0.710207915031993, 'Line 06 - 11': 0.7018502540337219, 'Line 10 - 11': 0.5930044655362867, 'Line 10 - 13': 0.58948327965074, 'Line 13 - 14': 0.7309132477824136}, location_weight_mass={'Bus 10': 0.17124912541676224, 'Bus 11': 0.12156315593742831, 'Bus 13': 0.12156315593742831, 'Line 06 - 11': 0.12156315593742831, 'Line 10 - 11': 0.17124912541676224, 'Line 10 - 13': 0.17124912541676224, 'Line 13 - 14': 0.12156315593742831}, neighborhood_density=41579.22908779595, n=584,

In [ ]:
def summary_value(summary, name: str, default=None):
    return getattr(summary, name, default) if summary is not None else default


df = pd.DataFrame(
    {
        "state": str(d["state"]),
        "cct_true": float(d["cct_true"]),
        "crit_gen_true": d["crit_gen_true"],
        "location_true": d["location_true"],
        "cct_weighted_per_location": d.get("cct_weighted_per_location"),
        "cct_weighted_global": d.get("cct_weighted_global", summary_value(d.get("prediction_summary"), "cct_weighted")),
        "has_crit_gen_prediction": d.get("has_crit_gen_prediction", d.get("prediction_summary") is not None),
        "has_location_prediction": d.get("has_location_prediction", d.get("cct_weighted_per_location") is not None),
        "location_weight_mass": d.get("location_weight_mass"),
        "location_neighbor_count": d.get("location_neighbor_count"),
        "n_neighbors": d.get("n_neighbors", summary_value(d.get("prediction_summary"), "n")),
        "n_eff": d.get("n_eff", summary_value(d.get("prediction_summary"), "n_eff")),
    }
    for d in data
)

df.head()

,state,cct_true,crit_gen_true,location_true,cct_weighted_per_location,cct_weighted_global,has_crit_gen_prediction,has_location_prediction,location_weight_mass,location_neighbor_count,n_neighbors,n_eff
0,13,0.55,g 03,bus 10,0.585491,0.646304,True,True,0.171249,100,584,580.142324
1,13,0.56,g 03,line 10 - 13,0.589483,0.646304,True,True,0.171249,100,584,580.142324
2,13,0.56,g 03,line 10 - 11,0.593004,0.646304,True,True,0.171249,100,584,580.142324
3,13,0.68,g 03,bus 13,0.710208,0.646304,True,True,0.121563,71,584,580.142324
4,13,0.70,g 03,line 13 - 14,0.730913,0.646304,True,True,0.121563,71,584,580.142324


In [ ]:
coverage = pd.Series(
    {
        "n_total": len(df),
        "n_with_crit_gen_prediction": int(df["has_crit_gen_prediction"].sum()),
        "n_with_location_prediction": int(df["has_location_prediction"].sum()),
        "crit_gen_coverage": float(df["has_crit_gen_prediction"].mean()),
        "location_coverage": float(df["has_location_prediction"].mean()),
        "n_missing_crit_gen_prediction": int((~df["has_crit_gen_prediction"]).sum()),
        "n_missing_location_prediction": int((~df["has_location_prediction"]).sum()),
    }
)

coverage

n_total                          1.044797e+06
n_with_crit_gen_prediction       1.044793e+06
n_with_location_prediction       1.044394e+06
crit_gen_coverage                9.999962e-01
location_coverage                9.996143e-01
n_missing_crit_gen_prediction    4.000000e+00
n_missing_location_prediction    4.030000e+02
dtype: float64

In [ ]:
missing = df.loc[~df["has_location_prediction"]].copy()
missing_by_crit_gen = missing["crit_gen_true"].value_counts()
missing_by_location = missing["location_true"].value_counts().head(20)

display(missing_by_crit_gen)
display(missing_by_location)

crit_gen_true
g 06    144
g 04     86
g 07     82
g 09     36
g 08     32
g 03     17
g 10      6
Name: count, dtype: int64

location_true
line 16 - 24    60
bus 24          47
line 15 - 16    34
bus 15          32
bus 16          24
line 16 - 21    22
bus 21          18
line 16 - 17    18
line 14 - 15    17
line 21 - 22    17
bus 02          12
line 02 - 03    10
line 01 - 02    10
bus 22          10
line 03 - 04     8
line 17 - 18     8
line 25 - 26     8
bus 17           7
bus 25           7
line 23 - 24     5
Name: count, dtype: int64

In [ ]:
def regression_metrics(frame: pd.DataFrame, pred_col: str) -> pd.Series:
    valid = frame[pred_col].notna()
    n_valid = int(valid.sum())

    if n_valid == 0:
        return pd.Series(
            {
                "n": 0,
                "coverage": 0.0,
                "mse": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "abs_err_min": np.nan,
                "abs_err_q25": np.nan,
                "abs_err_q50": np.nan,
                "abs_err_q75": np.nan,
                "abs_err_q90": np.nan,
                "abs_err_q95": np.nan,
                "abs_err_q99": np.nan,
                "abs_err_max": np.nan,
            }
        )

    y_true = frame.loc[valid, "cct_true"].to_numpy(dtype=float)
    y_pred = frame.loc[valid, pred_col].to_numpy(dtype=float)
    err = np.abs(y_pred - y_true)

    return pd.Series(
        {
            "n": n_valid,
            "coverage": float(valid.mean()),
            "mse": mean_squared_error(y_true, y_pred),
            "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
            "mae": mean_absolute_error(y_true, y_pred),
            "abs_err_min": np.quantile(err, 0.00),
            "abs_err_q25": np.quantile(err, 0.25),
            "abs_err_q50": np.quantile(err, 0.50),
            "abs_err_q75": np.quantile(err, 0.75),
            "abs_err_q90": np.quantile(err, 0.90),
            "abs_err_q95": np.quantile(err, 0.95),
            "abs_err_q99": np.quantile(err, 0.99),
            "abs_err_max": np.quantile(err, 1.00),
        }
    )


metrics_conditional = pd.DataFrame(
    {
        "per_location": regression_metrics(df, "cct_weighted_per_location"),
        "global_fallback": regression_metrics(df, "cct_weighted_global"),
    }
)

df["cct_pred_overall"] = df["cct_weighted_per_location"].fillna(df["cct_weighted_global"])
metrics_overall = pd.DataFrame(
    {
        "location_then_global": regression_metrics(df, "cct_pred_overall"),
    }
)

metrics_conditional, metrics_overall

(             per_location  global_fallback
 n            1.044394e+06     1.044793e+06
 coverage     9.996143e-01     9.999962e-01
 mse          8.367949e-04     1.675612e-02
 rmse         2.892741e-02     1.294454e-01
 mae          1.759688e-02     9.082011e-02
 abs_err_min  0.000000e+00     3.226308e-07
 abs_err_q25  5.575477e-03     3.015424e-02
 abs_err_q50  1.206626e-02     6.913945e-02
 abs_err_q75  2.197546e-02     1.223477e-01
 abs_err_q90  3.589999e-02     1.847214e-01
 abs_err_q95  4.914215e-02     2.357132e-01
 abs_err_q99  1.066439e-01     4.675123e-01
 abs_err_max  7.895555e-01     1.333293e+00,
              location_then_global
 n                    1.044793e+06
 coverage             9.999962e-01
 mse                  8.491903e-04
 rmse                 2.914087e-02
 mae                  1.762983e-02
 abs_err_min          0.000000e+00
 abs_err_q25          5.577392e-03
 abs_err_q50          1.207027e-02
 abs_err_q75          2.198704e-02
 abs_err_q90          3.593648e-0

In [ ]:
by_crit_gen = (
    df.groupby("crit_gen_true", observed=True)
    .apply(lambda g: regression_metrics(g, "cct_weighted_per_location"), include_groups=False)
    .sort_values("mae")
    .astype({"n": int})
)

by_crit_gen.to_csv("./bus39_report.csv")

by_crit_gen

,n,coverage,mse,rmse,mae,abs_err_min,abs_err_q25,abs_err_q50,abs_err_q75,abs_err_q90,abs_err_q95,abs_err_q99,abs_err_max
crit_gen_true,,,,,,,,,,,,,
g 04,50842,0.998311,0.000231,0.015215,0.010880,0.000000,0.004130,0.008641,0.014963,0.022082,0.027451,0.042186,0.417939
g 07,135791,0.999396,0.000325,0.018027,0.013245,0.000000,0.004917,0.010535,0.018476,0.027401,0.033865,0.050168,0.410619
g 06,96645,0.998512,0.000319,0.017863,0.013247,0.000000,0.005082,0.010810,0.018602,0.027224,0.033265,0.046721,0.428873
g 09,325510,0.999889,0.000371,0.019250,0.013529,0.000000,0.004748,0.010151,0.018099,0.028320,0.037079,0.062535,0.615180
g 05,21720,1.000000,0.000919,0.030309,0.022654,0.000003,0.008601,0.018167,0.031454,0.046504,0.057254,0.086663,0.511784
g 08,54450,0.999413,0.001762,0.041976,0.023128,0.000000,0.005481,0.012122,0.023492,0.056041,0.092473,0.177377,0.509432
g 03,357557,0.999952,0.001513,0.038894,0.023726,0.000000,0.007502,0.016198,0.029321,0.047740,0.067035,0.149816,0.789556
g 10,1879,0.996817,0.005211,0.072188,0.057305,0.000050,0.022535,0.047739,0.082368,0.118287,0.137152,0.194050,0.306896


In [ ]:
missing_diagnostics = {
    "missing_by_crit_gen": df.loc[~df["has_crit_gen_prediction"], "crit_gen_true"].value_counts().head(20),
    "missing_by_location": df.loc[~df["has_location_prediction"], "location_true"].value_counts().head(20),
}

missing_diagnostics

{'missing_by_crit_gen': crit_gen_true
 g 10    4
 Name: count, dtype: int64,
 'missing_by_location': location_true
 line 16 - 24    60
 bus 24          47
 line 15 - 16    34
 bus 15          32
 bus 16          24
 line 16 - 21    22
 bus 21          18
 line 16 - 17    18
 line 14 - 15    17
 line 21 - 22    17
 bus 02          12
 line 02 - 03    10
 line 01 - 02    10
 bus 22          10
 line 03 - 04     8
 line 17 - 18     8
 line 25 - 26     8
 bus 17           7
 bus 25           7
 line 23 - 24     5
 Name: count, dtype: int64}